# Practical No. 7 — Apply LSTM to Predict Stock Prices from Historical Data

## Aim
To implement a Long Short-Term Memory (LSTM) neural network for predicting future stock prices using historical stock market data.

## Objectives
- To understand time-series forecasting.
- To preprocess stock market data.
- To build an LSTM model for stock price prediction.
- To train and evaluate the model.
- To visualize actual and predicted stock prices.

## Theory
### Stock Price Prediction
Stock Price Prediction involves forecasting future stock values using historical stock market data such as **Open Price**, **High Price**, **Low Price**, **Close Price**, and **Volume**. Time-series forecasting helps investors analyze market trends and make informed decisions.

### What is LSTM?
LSTM (Long Short-Term Memory) is a special type of Recurrent Neural Network (RNN) designed to learn long-term dependencies in sequential data. Unlike traditional RNNs, LSTMs can remember important information for longer periods and avoid the vanishing gradient problem.

**Components:** Forget Gate, Input Gate, Cell State, Output Gate. These gates control information flow through the network.

**Why LSTM for stock prediction?** Handles sequential data efficiently; captures long-term market trends; learns temporal dependencies; provides better forecasting than traditional RNNs.

## Dataset
The Word document lists Yahoo Finance, NSE India, BSE India, and Kaggle Datasets as possible sources for historical stock data. The experiment reads **`stock_data.csv`** and uses its **`Close`** column.

**Sample dataset shown in the document (illustrative only):**

| Date | Open | High | Low | Close | Volume |
|---|---:|---:|---:|---:|---:|
| 01-01-2024 | 120 | 125 | 118 | 123 | 500000 |
| 02-01-2024 | 123 | 128 | 122 | 127 | 650000 |

The two sample rows alone are **not sufficient** for 60-step sequences. Upload a historical `stock_data.csv` containing **more than 61 records**, with a numeric `Close` column, before running the notebook. No real dataset was attached, so none is invented here.

## Software Requirements
- Python 3.x
- TensorFlow / Keras
- NumPy
- Pandas
- Matplotlib
- Scikit-Learn
- Google Colab / Jupyter Notebook (execution environment)

Google Colab usually has these installed. If an import fails, uncomment and run the following installation cell.

In [1]:
# Uncomment only if packages are missing in Google Colab.
# %pip install numpy pandas matplotlib scikit-learn tensorflow

## Algorithm
1. Load historical stock data.
2. Select Closing Price column.
3. Normalize the data.
4. Create time-series sequences.
5. Split into training and testing datasets.
6. Build LSTM architecture.
7. Train the model.
8. Predict stock prices.
9. Evaluate performance.
10. Plot actual vs predicted prices.

## Mathematical Representation
The hidden state is updated based on previous information:

\[h_t=f(h_{t-1},x_t)\]

where \(x_t\) = current input, \(h_t\) = hidden state, \(h_{t-1}\) = previous hidden state.

## Program — Step 1: Import Libraries

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

## Step 2: Load Dataset
In Google Colab, select **Files** in the left sidebar and upload `stock_data.csv`, or run the upload cell below. Use the file name given in the Word document.

In [3]:
from pathlib import Path

if not Path('stock_data.csv').exists():
    try:
        from google.colab import files
    except ImportError:
        print('Upload stock_data.csv into the notebook working directory before continuing.')
    else:
        print('Please select stock_data.csv from your computer.')
        uploaded = files.upload()

if not Path('stock_data.csv').exists():
    raise FileNotFoundError('stock_data.csv is required, as specified in DL-Practicals.docx.')

data = pd.read_csv('stock_data.csv')
print(data.head())
if 'Close' not in data.columns:
    raise ValueError("The CSV must have a 'Close' column, as specified in the Word document.")
if len(data) <= 61:
    raise ValueError('Provide more than 61 rows so 60-step sequences can be created and split.')
if data['Close'].isna().any():
    raise ValueError("The 'Close' column contains missing values; use a complete numeric column.")

Upload stock_data.csv into the notebook working directory before continuing.


FileNotFoundError: stock_data.csv is required, as specified in DL-Practicals.docx.

## Step 3: Select Closing Price

In [ ]:
dataset = pd.to_numeric(data['Close'], errors='raise').values
dataset = dataset.reshape(-1, 1)
print('Close-price array shape:', dataset.shape)

## Step 4: Normalize Data

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
dataset_scaled = scaler.fit_transform(dataset)
print('Scaled data shape:', dataset_scaled.shape)

## Step 5: Create Sequences

In [ ]:
X = []
y = []
for i in range(60, len(dataset_scaled)):
    X.append(dataset_scaled[i-60:i, 0])
    y.append(dataset_scaled[i, 0])
X = np.array(X)
y = np.array(y)
X = np.reshape(X, (X.shape[0], X.shape[1], 1))
print('X shape:', X.shape, '| y shape:', y.shape)

## Step 6: Split Dataset

In [ ]:
train_size = int(len(X) * 0.8)
X_train = X[:train_size]
y_train = y[:train_size]
X_test = X[train_size:]
y_test = y[train_size:]
print('Training samples:', len(X_train), '| Testing samples:', len(X_test))

## Step 7: Build LSTM Model

In [ ]:
model = Sequential()
model.add(LSTM(units=50, return_sequences=True, input_shape=(X_train.shape[1], 1)))
model.add(LSTM(units=50))
model.add(Dense(25))
model.add(Dense(1))

## Step 8: Compile Model

In [ ]:
model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

## Step 9: Train Model

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32
)

## Step 10: Predict Stock Prices

In [ ]:
predictions = model.predict(X_test)
predictions = scaler.inverse_transform(predictions)
print('Predictions shape:', predictions.shape)

## Step 11: Evaluate Model

In [ ]:
from sklearn.metrics import mean_squared_error

rmse = np.sqrt(mean_squared_error(
    scaler.inverse_transform(y_test.reshape(-1, 1)),
    predictions
))
print('RMSE:', rmse)

## Step 12: Visualization

In [ ]:
actual_prices = scaler.inverse_transform(y_test.reshape(-1, 1))
plt.figure(figsize=(10, 5))
plt.plot(actual_prices, label='Actual Price')
plt.plot(predictions, label='Predicted Price')
plt.title('Stock Price Prediction Using LSTM')
plt.xlabel('Time')
plt.ylabel('Stock Price')
plt.legend()
plt.show()

## Expected Output (from Word document — not actual executed results)

**Model summary (layers):**

| Layer (type) | Output shape |
|---|---|
| LSTM | (None, 60, 50) |
| LSTM | (None, 50) |
| Dense | (None, 25) |
| Dense | (None, 1) |

**Training output (illustrative):**

```text
Epoch 1/20   Loss: 0.0215
Epoch 20/20  Loss: 0.0012
```

**Evaluation output (illustrative):** `RMSE: 2.45`

**Prediction graph:** Actual Price Line and Predicted Price Line (both lines closely overlap in the document's expected description).

**Note:** Actual loss, RMSE, plots and model summary depend on the uploaded stock data, installed TensorFlow version, and model training. These outputs are not precomputed or claimed as tested.

## Result (from Word document)
An LSTM-based deep learning model was successfully implemented to predict stock prices from historical data. The model learned temporal patterns from previous stock prices and generated future price predictions with low prediction error.

*Run Steps 1–12 with your own `stock_data.csv` to confirm the result for your dataset.*